Обработка интерференционных изображений методом двойного Фурье-преобразования.

In [1]:
import os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy import ndimage as ndi
from scipy.signal import find_peaks
from skimage.restoration import unwrap_phase

In [2]:
IMAGE_PATH = r"/content/1.bmp"
OUT_DIR = Path(r"/content/result_1")
CROPS_DIR = OUT_DIR / "cropped"

PIXEL_SIZE_SENSOR_UM = 2.2          # размер пикселя камеры
MICROSCOPE_MAGNIFICATION = 1.0
WAVELENGTH_UM = 0.65
DELTA_N = None                      # разность показателей преломления (n_cell - n_medium)
                                    # Если None, строится карта оптической толщины OPD

FRINGE_SUPPRESS_SIGMA = 5.0         # сглаживание для поиска клеток на исходной интерферограмме
MIN_CELL_RADIUS_PX = 30
MAX_CELL_RADIUS_PX = 90
MIN_DIST_BETWEEN_CELLS_PX = 55
MAX_CELLS_TO_SAVE = 20

# параметры спектрального фильтра для двойного Фурье
CENTER_EXCLUSION_RADIUS = 60        # область вокруг DC-компоненты
LOBE_RADIUS = 70                    # радиус выделяемого спектрального плеча

PLOT_DPI = 160

In [3]:
USE_SPHERICAL_DETREND = True

# радиусы кривизны в пикселях (None оцениваются автоматически по фазе)
SPHERICAL_RX_PX = None
SPHERICAL_RY_PX = None

# учитывать ли линейные члены
FIT_TILT_AND_OFFSET = False

In [4]:
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def read_gray_image(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Не удалось прочитать изображение: {path}")
    return img

def pixel_size_object_um() -> float:
    return PIXEL_SIZE_SENSOR_UM / MICROSCOPE_MAGNIFICATION

def fft2c(img: np.ndarray) -> np.ndarray:
    return np.fft.fftshift(np.fft.fft2(img))

def ifft2c(spec: np.ndarray) -> np.ndarray:
    return np.fft.ifft2(np.fft.ifftshift(spec))

def log_spectrum(F: np.ndarray) -> np.ndarray:
    return np.log1p(np.abs(F))

In [5]:
def circular_mask(shape, center, radius):
    h, w = shape
    yy, xx = np.ogrid[:h, :w]
    cy, cx = center
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2

def find_first_order_peak(img: np.ndarray, center_exclusion_radius=CENTER_EXCLUSION_RADIUS):
    F = fft2c(img.astype(np.float32))
    mag = log_spectrum(F)
    h, w = img.shape
    cy, cx = h // 2, w // 2

    mag_search = mag.copy()
    mag_search[circular_mask(img.shape, (cy, cx), center_exclusion_radius)] = 0

    peak = np.unravel_index(np.argmax(mag_search), mag_search.shape)
    py, px = peak
    return F, mag, (py, px)

In [6]:
def unwrap_phase_2d(phase_wrapped: np.ndarray) -> np.ndarray:
    """
    Устойчивая развёртка фазы
    """
    return unwrap_phase(phase_wrapped)

def detrend_phase(phase_unwrapped: np.ndarray) -> np.ndarray:
    """
    Убираем наклон/фон плоскостью z = ax + by + c
    """
    h, w = phase_unwrapped.shape
    yy, xx = np.mgrid[:h, :w]
    A = np.column_stack([xx.ravel(), yy.ravel(), np.ones(h * w)])
    b = phase_unwrapped.ravel()
    coeff, *_ = np.linalg.lstsq(A, b, rcond=None)
    plane = (coeff[0] * xx + coeff[1] * yy + coeff[2])
    return phase_unwrapped - plane

In [7]:
def build_background_mask(amplitude: np.ndarray,
                          blur_sigma: float = 6.0,
                          q: float = 0.55) -> np.ndarray:
    # q — квантиль для отбора пикселей фона по сглаженной амплитуде
    amp = amplitude.astype(np.float32)
    amp_smooth = ndi.gaussian_filter(amp, blur_sigma)

    thr = np.quantile(amp_smooth, q)
    bg_mask = amp_smooth <= thr

    bg_mask = ndi.binary_opening(bg_mask, structure=np.ones((5, 5)))
    bg_mask = ndi.binary_closing(bg_mask, structure=np.ones((7, 7)))
    bg_mask = ndi.binary_fill_holes(bg_mask)

    return bg_mask

In [8]:
def fit_quadratic_background_robust(phase_unwrapped: np.ndarray,
                                    mask: np.ndarray = None,
                                    max_iter: int = 3,
                                    sigma_clip: float = 2.5,
                                    fit_tilt: bool = True):
    """
    Аппроксимация гладкого фона квадратичной моделью:
        phi(x, y) = a*X^2 + b*Y^2 + c*X*Y + d*X + e*Y + f
    """
    h, w = phase_unwrapped.shape
    yy, xx = np.mgrid[:h, :w].astype(np.float64)

    x0 = (w - 1) / 2.0
    y0 = (h - 1) / 2.0
    X = xx - x0
    Y = yy - y0

    valid = np.isfinite(phase_unwrapped)
    if mask is not None:
        valid &= mask

    if valid.sum() < 50:
        raise ValueError("Слишком мало точек для оценки сферического тренда")

    Z = phase_unwrapped.copy()

    current_mask = valid.copy()

    for _ in range(max_iter):
        if fit_tilt:
            A = np.column_stack([
                X[current_mask] ** 2,
                Y[current_mask] ** 2,
                X[current_mask] * Y[current_mask],
                X[current_mask],
                Y[current_mask],
                np.ones(current_mask.sum(), dtype=np.float64),
            ])
        else:
            A = np.column_stack([
                X[current_mask] ** 2,
                Y[current_mask] ** 2,
                X[current_mask] * Y[current_mask],
                np.ones(current_mask.sum(), dtype=np.float64),
            ])

        b = Z[current_mask]
        coeff, *_ = np.linalg.lstsq(A, b, rcond=None)

        if fit_tilt:
            a, b2, c, d, e, f0 = coeff
            trend = a * X**2 + b2 * Y**2 + c * X * Y + d * X + e * Y + f0
            params = {
                "a": a, "b": b2, "c": c, "d": d, "e": e, "f": f0,
                "x0": x0, "y0": y0
            }
        else:
            a, b2, c, f0 = coeff
            trend = a * X**2 + b2 * Y**2 + c * X * Y + f0
            params = {
                "a": a, "b": b2, "c": c, "d": 0.0, "e": 0.0, "f": f0,
                "x0": x0, "y0": y0
            }

        resid = Z - trend
        med = np.median(resid[current_mask])
        mad = np.median(np.abs(resid[current_mask] - med)) + 1e-12
        robust_sigma = 1.4826 * mad

        new_mask = valid & (np.abs(resid - med) < sigma_clip * robust_sigma)

        # чтобы не потерять фон
        if mask is not None:
            new_mask &= mask

        if new_mask.sum() < 100:
            break

        current_mask = new_mask

    return trend, params, current_mask

In [9]:
def spherical_phase_multiplier(shape, params):
    """
    Строит фазовый множитель exp(-i * phi_trend)
    """
    h, w = shape
    yy, xx = np.mgrid[:h, :w].astype(np.float64)

    x0 = params["x0"]
    y0 = params["y0"]
    X = xx - x0
    Y = yy - y0

    phi = (
        params["a"] * X**2 +
        params["b"] * Y**2 +
        params["c"] * X * Y +
        params["d"] * X +
        params["e"] * Y +
        params["f"]
    )

    return np.exp(-1j * phi), phi

def remove_spherical_trend_robust(complex_field: np.ndarray,
                                  amplitude: np.ndarray = None,
                                  background_mask: np.ndarray = None,
                                  fit_tilt: bool = True):
    wrapped_phase = np.angle(complex_field)
    unwrapped_phase = unwrap_phase_2d(wrapped_phase)

    if amplitude is None:
        amplitude = np.abs(complex_field)

    if background_mask is None:
        background_mask = build_background_mask(amplitude)

    trend, params, fit_mask = fit_quadratic_background_robust(
        unwrapped_phase,
        mask=background_mask,
        max_iter=3,
        sigma_clip=2.5,
        fit_tilt=fit_tilt
    )

    phase_mult, trend_map = spherical_phase_multiplier(complex_field.shape, params)
    corrected_field = complex_field * phase_mult

    corrected_wrapped = np.angle(corrected_field)
    corrected_unwrapped = unwrap_phase_2d(corrected_wrapped)

    return {
        "corrected_field": corrected_field,
        "original_wrapped_phase": wrapped_phase,
        "original_unwrapped_phase": unwrapped_phase,
        "background_mask": background_mask,
        "fit_mask": fit_mask,
        "spherical_trend": trend_map,
        "corrected_wrapped_phase": corrected_wrapped,
        "corrected_unwrapped_phase": corrected_unwrapped,
        "params": params,
    }

In [10]:
def reconstruct_complex_field_double_fft(
        img: np.ndarray,
        lobe_radius=LOBE_RADIUS,
        center_exclusion_radius=CENTER_EXCLUSION_RADIUS,
        include_dc=False,
        margin_pct=0.0,
        use_spherical_detrend=USE_SPHERICAL_DETREND,
        fit_tilt=FIT_TILT_AND_OFFSET):
    F, mag, (py, px) = find_first_order_peak(img, center_exclusion_radius)

    h, w = img.shape
    cy, cx = h // 2, w // 2

    # базовый радиус
    r = int(lobe_radius * (1 + margin_pct))
    # маска первого порядка
    mask = circular_mask(img.shape, (py, px), r)

    if include_dc:
        # добавить центральный порядок
        mask |= circular_mask(img.shape, (cy, cx), center_exclusion_radius)
    else:
        # вырезать DC
        mask &= ~circular_mask(img.shape, (cy, cx), center_exclusion_radius)

    mask = mask.astype(np.float32)
    sideband = F * mask

    shift_y = cy - py
    shift_x = cx - px
    sideband_shifted = np.roll(sideband, shift=(shift_y, shift_x), axis=(0, 1))

    complex_field = ifft2c(sideband_shifted)
    amplitude = np.abs(complex_field)
    wrapped_phase = np.angle(complex_field)
    unwrapped_phase = unwrap_phase_2d(wrapped_phase)

    corrected_field = complex_field
    corrected_wrapped_phase = wrapped_phase
    corrected_unwrapped_phase = unwrapped_phase
    spherical_trend = np.zeros_like(unwrapped_phase)
    background_mask = np.zeros_like(unwrapped_phase, dtype=bool)
    fit_mask = np.zeros_like(unwrapped_phase, dtype=bool)
    spherical_params = None

    if use_spherical_detrend:
        background_mask = build_background_mask(amplitude)

        detr = remove_spherical_trend_robust(
            complex_field,
            amplitude=amplitude,
            background_mask=background_mask,
            fit_tilt=fit_tilt
        )

        corrected_field = detr["corrected_field"]
        corrected_wrapped_phase = detr["corrected_wrapped_phase"]
        corrected_unwrapped_phase = detr["corrected_unwrapped_phase"]
        spherical_trend = detr["spherical_trend"]
        background_mask = detr["background_mask"]
        fit_mask = detr["fit_mask"]
        spherical_params = detr["params"]

    return {
        "F": F,
        "log_spectrum": mag,
        "peak": (py, px),
        "mask": mask,
        "sideband": sideband,
        "sideband_shifted": sideband_shifted,
        "complex_field": complex_field,
        "amplitude": amplitude,
        "wrapped_phase": wrapped_phase,
        "unwrapped_phase": unwrapped_phase,
        "corrected_field": corrected_field,
        "corrected_wrapped_phase": corrected_wrapped_phase,
        "corrected_unwrapped_phase": corrected_unwrapped_phase,
        "spherical_trend": spherical_trend,
        "background_mask": background_mask,
        "fit_mask": fit_mask,
        "spherical_params": spherical_params,
    }

In [11]:
def phase_to_opd_um(phase: np.ndarray) -> np.ndarray:
    """
    Переводит фазу в OPD
    """
    return (WAVELENGTH_UM * phase) / (2 * np.pi)

def opd_to_height_um(opd_um: np.ndarray) -> np.ndarray:
    """
    Для прозрачного объекта в проходящем свете:
    opd = (n_obj - n_medium) * h = delta_n * h
    h = opd / delta_n
    """
    if DELTA_N is None:
        return opd_um
    if DELTA_N == 0:
        raise ValueError("DELTA_N не может быть 0")
    return opd_um / DELTA_N

In [12]:
def suppress_fringes_for_segmentation(img: np.ndarray) -> np.ndarray:
    """
    Делаем сглаживание изобрадения, чтобы границы эритроцитов стали заметнее
    """
    blur = ndi.gaussian_filter(img.astype(np.float32), FRINGE_SUPPRESS_SIGMA)
    blur = cv2.normalize(blur, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return blur

def detect_cells_hough(img_gray: np.ndarray):
    """
    Ищет эритроциты методом Хафа как окружности
    """
    smooth = suppress_fringes_for_segmentation(img_gray)
    circles = cv2.HoughCircles(
        smooth,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=MIN_DIST_BETWEEN_CELLS_PX,
        param1=80,
        param2=18,
        minRadius=MIN_CELL_RADIUS_PX,
        maxRadius=MAX_CELL_RADIUS_PX,
    )
    if circles is None:
        return []

    circles = np.round(circles[0]).astype(int)
    circles = filter_circles(img_gray, circles)
    return circles

In [13]:
def circle_quality_score(img: np.ndarray, x: int, y: int, r: int) -> float:
    """
    Оценивает качество найденных клеток
    """
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
    d = np.sqrt((xx - x) ** 2 + (yy - y) ** 2)

    ring = (d >= r - 4) & (d <= r + 4)
    inner = d <= max(3, int(0.55 * r))
    outer = (d >= r + 8) & (d <= r + 18)

    if ring.sum() == 0 or inner.sum() == 0 or outer.sum() == 0:
        return np.inf

    ring_mean = float(img[ring].mean())
    inner_mean = float(img[inner].mean())
    outer_mean = float(img[outer].mean())
    return ring_mean - 0.5 * inner_mean - 0.5 * outer_mean

def filter_circles(img: np.ndarray, circles: np.ndarray):
    scored = []
    h, w = img.shape
    for x, y, r in circles:
        if x - r < 2 or y - r < 2 or x + r >= w - 2 or y + r >= h - 2:
            continue
        score = circle_quality_score(img, x, y, r)
        if not np.isfinite(score):
            continue
        scored.append((score, x, y, r))

    scored.sort(key=lambda t: t[0])  # лучшие в начале
    selected = []
    for score, x, y, r in scored:
        # порог, чтобы убрать шум
        if score > -2.0:
            continue

        overlap = False
        for _, sx, sy, sr in selected:
            dist = np.hypot(x - sx, y - sy)
            if dist < 0.8 * (r + sr):
                overlap = True
                break
        if not overlap:
            selected.append((score, x, y, r))

    selected = selected[:MAX_CELLS_TO_SAVE]
    return [(x, y, r, score) for score, x, y, r in selected]

In [14]:
def crop_and_save_cells(img: np.ndarray, cells, out_dir: Path):
    ensure_dir(out_dir)
    saved = []
    h, w = img.shape
    for idx, (x, y, r, score) in enumerate(cells, start=1):
        pad = int(0.35 * r)
        x1 = max(0, x - r - pad)
        y1 = max(0, y - r - pad)
        x2 = min(w, x + r + pad)
        y2 = min(h, y + r + pad)
        crop = img[y1:y2, x1:x2]
        crop_path = out_dir / f"rbc_{idx:02d}_x{x}_y{y}_r{r}.png"
        cv2.imwrite(str(crop_path), crop)
        saved.append({
            "id": idx,
            "center_x": x,
            "center_y": y,
            "radius_px": r,
            "score": score,
            "path": crop_path,
            "bbox": (x1, y1, x2, y2),
        })
    return saved

def choose_main_cell(cells_info):
    """
    Выбираем наиболее центральную и качественную клетку
    """
    if not cells_info:
        raise RuntimeError("Не найдено ни одного эритроцита")
    centers = np.array([[c["center_x"], c["center_y"]] for c in cells_info], dtype=float)
    center_img = centers.mean(axis=0)
    best = sorted(
        cells_info,
        key=lambda c: (np.hypot(c["center_x"] - center_img[0], c["center_y"] - center_img[1]), abs(c["radius_px"] - 50))
    )[0]
    return best

In [15]:
def segment_cell_in_crop(crop_phase: np.ndarray):
    """
    Сегментация по карте фазы/OPD:
    - сглаживание
    - Otsu
    - морфологическая очистка
    """
    phase_sm = ndi.gaussian_filter(crop_phase.astype(np.float32), 2.0)
    phase_norm = cv2.normalize(phase_sm, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    _, mask = cv2.threshold(phase_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Проверяем, не выбрали ли фон
    if mask.mean() > 127:
        mask = 255 - mask

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Берём крупнейшую связную компоненту ближе к центру
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)
    if num_labels <= 1:
        return mask.astype(bool)

    h, w = mask.shape
    img_center = np.array([w / 2, h / 2], dtype=float)

    best_label = None
    best_cost = np.inf
    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area < 150:
            continue
        cxy = centroids[lab]
        dist = np.linalg.norm(cxy - img_center)
        cost = dist - 0.02 * area
        if cost < best_cost:
            best_cost = cost
            best_label = lab

    if best_label is None:
        return mask.astype(bool)

    return labels == best_label

In [16]:
def plot_intermediate_results(img: np.ndarray, recon: dict, out_dir: Path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0, 0].imshow(img, cmap="gray")
    axes[0, 0].set_title("Исходная интерферограмма")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(recon["log_spectrum"], cmap="gray")
    py, px = recon["peak"]
    axes[0, 1].plot(px, py, "ro", markersize=6)
    axes[0, 1].set_title("Логарифм спектра БПФ")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(recon["mask"], cmap="gray")
    axes[0, 2].set_title("Маска выбранного спектрального плеча")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(np.log1p(np.abs(recon["sideband_shifted"])), cmap="gray")
    axes[1, 0].set_title("Плечо после переноса к центру")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(recon["amplitude"], cmap="gray")
    axes[1, 1].set_title("Амплитуда после 2-го БПФ")
    axes[1, 1].axis("off")

    phase_to_show = recon["corrected_unwrapped_phase"]
    im = axes[1, 2].imshow(phase_to_show, cmap="jet")
    axes[1, 2].set_title("Фаза после сферического детренда")
    axes[1, 2].axis("off")
    fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    fig.savefig(out_dir / "01_intermediate_fft.png", dpi=PLOT_DPI)
    plt.close(fig)

In [45]:
def plot_detected_cells(img, cells, out_dir: Path):
    ensure_dir(out_dir)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for i, (x, y, r, score) in enumerate(cells, start=1):
        cv2.circle(vis, (x, y), r, (0, 255, 0), 2)
        cv2.putText(vis, str(i), (x - 10, y + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 0, 0), 2)
    cv2.imwrite(str(out_dir / "02_detected_cells.png"), vis)

def plot_main_cell_3d(z_um: np.ndarray, mask: np.ndarray, out_path: Path, title: str):
    h, w = z_um.shape
    px_um = pixel_size_object_um()
    x = np.arange(w) * px_um
    y = np.arange(h) * px_um
    X, Y = np.meshgrid(x, y)

    Z = z_um.copy()
    Z[~mask] = np.nan

    fig = plt.figure(figsize=(12, 7))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(X, Y, Z, cmap=cm.viridis, linewidth=0, antialiased=True)

    ax.set_xlabel("X, мкм")
    ax.set_ylabel("Y, мкм")
    ax.set_zlabel("Z, мкм")
    ax.set_title(title)
    ax.view_init(elev=15, azim=100)

    cbar = fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.08)
    cbar.set_label("Высота, мкм" if DELTA_N is not None else "OPD, мкм")

    fig.tight_layout()
    fig.savefig(out_path, dpi=PLOT_DPI)
    plt.close(fig)

In [18]:
def plot_result_for_config(result, title, save_path=None):
    F = result["F"]
    peak = result["peak"]
    mask = result["mask"]
    phase = result["unwrapped_phase"]

    fig = plt.figure(figsize=(12, 4))

    ax1 = fig.add_subplot(1, 2, 1)
    ax1.imshow(mask, cmap="gray")
    ax1.set_title("Окно фильтра")
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 2, 2)
    im = ax2.imshow(phase, cmap="jet")
    ax2.set_title("Фаза после детренда")
    ax2.set_xlabel("X, px")
    ax2.set_ylabel("Y, px")

    cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label("Phase, rad")

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")

    plt.close(fig)

In [39]:
IMAGE_PATH = r"/content/3.bmp"
OUT_DIR = Path(r"/content/result_3")

In [43]:
IMAGE_PATH = r"/content/4.bmp"
OUT_DIR = Path(r"/content/result_4")

PIXEL_SIZE_SENSOR_UM = 2.2          # размер пикселя камеры
MICROSCOPE_MAGNIFICATION = 1.0
WAVELENGTH_UM = 0.55
DELTA_N = None                      # разность показателей преломления (n_cell - n_medium)
                                    # Если None, строится карта оптической толщины OPD

FRINGE_SUPPRESS_SIGMA = 1.0
MIN_CELL_RADIUS_PX = 25
MAX_CELL_RADIUS_PX = 70
MIN_DIST_BETWEEN_CELLS_PX = 45
MAX_CELLS_TO_SAVE = 20

CENTER_EXCLUSION_RADIUS = 30
LOBE_RADIUS = 40

PLOT_DPI = 160

In [46]:
ensure_dir(OUT_DIR)
ensure_dir(CROPS_DIR)

img = read_gray_image(IMAGE_PATH)

recon = reconstruct_complex_field_double_fft(
    img,
    use_spherical_detrend=True,
    fit_tilt=True
)

phase_after_spherical = recon["corrected_unwrapped_phase"]
phase_flat = detrend_phase(phase_after_spherical)

opd_um = phase_to_opd_um(phase_flat)
z_um = opd_to_height_um(opd_um)

# Поиск эритроцитов
cells = detect_cells_hough(img)
plot_detected_cells(img, cells, OUT_DIR)
cells_info = crop_and_save_cells(img, cells, CROPS_DIR)

# Выбор одного основного эритроцита
main_cell = choose_main_cell(cells_info)
x1, y1, x2, y2 = main_cell["bbox"]

crop_img = img[y1:y2, x1:x2]
crop_phase = phase_flat[y1:y2, x1:x2]
crop_opd = opd_um[y1:y2, x1:x2]
crop_z = z_um[y1:y2, x1:x2]

cell_mask = segment_cell_in_crop(crop_opd)

# Вычитаем фон по области вне клетки
if np.any(~cell_mask):
    background_level = np.median(crop_z[~cell_mask])
    crop_z = crop_z - background_level

# Визуализации
plot_intermediate_results(img, recon, OUT_DIR)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(crop_img, cmap="gray")
axes[0].set_title("Основной вырезанный эритроцит")
axes[0].axis("off")

im1 = axes[1].imshow(crop_opd, cmap="viridis")
axes[1].set_title("Карта OPD")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(OUT_DIR / "03_main_cell_maps.png", dpi=PLOT_DPI)
plt.close(fig)

plot_main_cell_3d(
    crop_z,
    cell_mask,
    OUT_DIR / "04_main_cell_3d.png",
    "3D-профиль эритроцита"
)

In [53]:
import cv2

def read_color_image(filename):
    img = cv2.imread(filename, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img is None:
        raise ValueError(f"Не удалось загрузить изображение: {filename}")
    return img

fst = read_color_image('/content/Img1 — копия.bmp')
snd = read_color_image('/content/44б — копия.bmp')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(fst)
axes[0].set_title("Исходная интерферограмма 1")
axes[0].axis("off")

im1 = axes[1].imshow(snd)
axes[1].set_title("Исходная интерферограмма 2")
axes[1].axis("off")

fig.tight_layout()
fig.savefig("dark_spots_example.png", dpi=PLOT_DPI)
plt.close(fig)

In [ ]:
configs = [
    {
        "name": "same_size",
        "lobe_radius": LOBE_RADIUS,
        "center_exclusion_radius": CENTER_EXCLUSION_RADIUS,
        "include_dc": False,
        "margin_pct": 0.0,
    },
    {
        "name": "small",
        "lobe_radius": 8,
        "center_exclusion_radius": CENTER_EXCLUSION_RADIUS,
        "include_dc": False,
        "margin_pct": 0.0,
    },
    {
        "name": "first_and_zero",
        "lobe_radius": LOBE_RADIUS,
        "center_exclusion_radius": CENTER_EXCLUSION_RADIUS,
        "include_dc": True,
        "margin_pct": 0.0,
    },
    {
        "name": "first_no_zero",
        "lobe_radius": LOBE_RADIUS,
        "center_exclusion_radius": CENTER_EXCLUSION_RADIUS,
        "include_dc": False,
        "margin_pct": 0.05,
    },
]

In [ ]:
# разные окна фильтра
img = read_gray_image(IMAGE_PATH)

compare_dir = OUT_DIR / "different_filter_window"
ensure_dir(compare_dir)

for cfg in configs:
    result = reconstruct_complex_field_double_fft(
        img=img,
        lobe_radius=cfg["lobe_radius"],
        center_exclusion_radius=cfg["center_exclusion_radius"],
        include_dc=cfg["include_dc"],
        margin_pct=cfg["margin_pct"],
    )

    save_path = compare_dir / f"{cfg['name']}.png"
    plot_result_for_config(result, cfg["name"], save_path=save_path)

In [ ]:
IMAGE_PATH = r"/content/3.bmp"
OUT_DIR = Path(r"/content/result_3")

ensure_dir(OUT_DIR)
ensure_dir(CROPS_DIR)

img = read_gray_image(IMAGE_PATH)

recon = reconstruct_complex_field_double_fft(
    img,
    use_spherical_detrend=True,
    fit_tilt=True
)

phase_after_spherical = recon["corrected_unwrapped_phase"]
phase_flat = detrend_phase(phase_after_spherical)

opd_um = phase_to_opd_um(phase_flat)
z_um = opd_to_height_um(opd_um)

# Поиск эритроцитов
cells = detect_cells_hough(img)
plot_detected_cells(img, cells, OUT_DIR)
cells_info = crop_and_save_cells(img, cells, CROPS_DIR)

# Выбор одного основного эритроцита
main_cell = choose_main_cell(cells_info)
x1, y1, x2, y2 = main_cell["bbox"]

crop_img = img[y1:y2, x1:x2]
crop_phase = phase_flat[y1:y2, x1:x2]
crop_opd = opd_um[y1:y2, x1:x2]
crop_z = z_um[y1:y2, x1:x2]

cell_mask = segment_cell_in_crop(crop_opd)

# Вычитаем фон по области вне клетки
if np.any(~cell_mask):
    background_level = np.median(crop_z[~cell_mask])
    crop_z = crop_z - background_level

# Визуализации
plot_intermediate_results(img, recon, OUT_DIR)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(crop_img, cmap="gray")
axes[0].set_title("Основной вырезанный эритроцит")
axes[0].axis("off")

im1 = axes[1].imshow(crop_opd, cmap="viridis")
axes[1].set_title("Карта OPD")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(OUT_DIR / "03_main_cell_maps.png", dpi=PLOT_DPI)
plt.close(fig)

plot_main_cell_3d(
    crop_z,
    cell_mask,
    OUT_DIR / "04_main_cell_3d.png",
    "3D-профиль эритроцита"
)

# разные окна фильтра
img = read_gray_image(IMAGE_PATH)

compare_dir = OUT_DIR / "different_filter_window"
ensure_dir(compare_dir)

for cfg in configs:
    result = reconstruct_complex_field_double_fft(
        img=img,
        lobe_radius=cfg["lobe_radius"],
        center_exclusion_radius=cfg["center_exclusion_radius"],
        include_dc=cfg["include_dc"],
        margin_pct=cfg["margin_pct"],
    )

    save_path = compare_dir / f"{cfg['name']}.png"
    plot_result_for_config(result, cfg["name"], save_path=save_path)

In [ ]:
IMAGE_PATH = r"/content/4.bmp"
OUT_DIR = Path(r"/content/result_4")

PIXEL_SIZE_SENSOR_UM = 2.2          # размер пикселя камеры
MICROSCOPE_MAGNIFICATION = 1.0
WAVELENGTH_UM = 0.55
DELTA_N = None                      # разность показателей преломления (n_cell - n_medium)
                                    # Если None, строится карта оптической толщины OPD

FRINGE_SUPPRESS_SIGMA = 1.0
MIN_CELL_RADIUS_PX = 25
MAX_CELL_RADIUS_PX = 70
MIN_DIST_BETWEEN_CELLS_PX = 45
MAX_CELLS_TO_SAVE = 20

CENTER_EXCLUSION_RADIUS = 30
LOBE_RADIUS = 40

PLOT_DPI = 160

ensure_dir(OUT_DIR)
ensure_dir(CROPS_DIR)

img = read_gray_image(IMAGE_PATH)

recon = reconstruct_complex_field_double_fft(
    img,
    use_spherical_detrend=True,
    fit_tilt=True
)

phase_after_spherical = recon["corrected_unwrapped_phase"]
phase_flat = detrend_phase(phase_after_spherical)

opd_um = phase_to_opd_um(phase_flat)
z_um = opd_to_height_um(opd_um)

# Поиск эритроцитов
cells = detect_cells_hough(img)
plot_detected_cells(img, cells, OUT_DIR)
cells_info = crop_and_save_cells(img, cells, CROPS_DIR)

# Выбор одного основного эритроцита
main_cell = choose_main_cell(cells_info)
x1, y1, x2, y2 = main_cell["bbox"]

crop_img = img[y1:y2, x1:x2]
crop_phase = phase_flat[y1:y2, x1:x2]
crop_opd = opd_um[y1:y2, x1:x2]
crop_z = z_um[y1:y2, x1:x2]

cell_mask = segment_cell_in_crop(crop_opd)

# Вычитаем фон по области вне клетки
if np.any(~cell_mask):
    background_level = np.median(crop_z[~cell_mask])
    crop_z = crop_z - background_level

# Визуализации
plot_intermediate_results(img, recon, OUT_DIR)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(crop_img, cmap="gray")
axes[0].set_title("Основной вырезанный эритроцит")
axes[0].axis("off")

im1 = axes[1].imshow(crop_opd, cmap="viridis")
axes[1].set_title("Карта OPD")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(OUT_DIR / "03_main_cell_maps.png", dpi=PLOT_DPI)
plt.close(fig)

plot_main_cell_3d(
    crop_z,
    cell_mask,
    OUT_DIR / "04_main_cell_3d.png",
    "3D-профиль эритроцита"
)

# разные окна фильтра
img = read_gray_image(IMAGE_PATH)

compare_dir = OUT_DIR / "different_filter_window"
ensure_dir(compare_dir)

for cfg in configs:
    result = reconstruct_complex_field_double_fft(
        img=img,
        lobe_radius=cfg["lobe_radius"],
        center_exclusion_radius=cfg["center_exclusion_radius"],
        include_dc=cfg["include_dc"],
        margin_pct=cfg["margin_pct"],
    )

    save_path = compare_dir / f"{cfg['name']}.png"
    plot_result_for_config(result, cfg["name"], save_path=save_path)